# Train Cogsworth on Kaggle

Fine-tunes a small open model (Qwen2.5 0.5B Instruct by default) with LoRA on Cogsworth's grounded
training data, then exports browser-ready weights for transformers.js.

**Before you run:**
1. Upload the kit as a Kaggle dataset: zip `cogsworth/train/data` and `cogsworth/train/python`
   (see the README) and add it to this notebook with **Add Input**. Its name doesn't matter, the next
   cell finds it.
2. **Settings → Accelerator → GPU T4** (one T4 is enough; with T4 x2 the notebook uses one).
3. **Settings → Internet → On** (needs a phone-verified Kaggle account). The base model and the
   Python packages download from the internet.
4. **Run All.** With the 0.5B model the whole notebook takes roughly an hour, most of it training.

Everything you need afterwards lands in `/kaggle/working/cogsworth-out.zip` (Output tab → download).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4; must be set before torch loads
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE = "Qwen/Qwen2.5-0.5B-Instruct"   # or "Qwen/Qwen2.5-1.5B-Instruct": better answers, ~3x slower to train, ~1.5 GB in the browser
EPOCHS = 3
EXPORT_FORMATS = "q8,q4"              # q8 (~0.5 GB for 0.5B) is the one to ship; q4 keeps the big embedding table in fp32, so it ends up larger for Qwen
REPO_BRANCH = "main"                  # only used if no dataset is attached
for k, v in {"BASE": BASE, "EPOCHS": str(EPOCHS), "EXPORT_FORMATS": EXPORT_FORMATS}.items():
    os.environ[k] = v   # the shell cells below read these

In [ ]:
# Find the uploaded kit (any folder under /kaggle/input that holds data/train.jsonl and python/train_lora.py)
import glob, shutil, subprocess
from pathlib import Path

WORK = Path("/kaggle/working/cogsworth-train")
hits = [Path(p).parent.parent for p in glob.glob("/kaggle/input/**/data/train.jsonl", recursive=True)]
hits = [h for h in hits if (h / "python" / "train_lora.py").exists()]
if WORK.exists():
    shutil.rmtree(WORK)
REPO = None
if hits:
    print("Using dataset:", hits[0])
    if (hits[0].parent.parent / "portfolio-data.js").exists():
        REPO = hits[0].parent.parent   # a GitHub-repo dataset holds the whole site, which the scoring step needs
    shutil.copytree(hits[0], WORK, ignore=shutil.ignore_patterns("__pycache__", "out"))
else:
    print("No dataset attached, cloning the kit from GitHub instead")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                    "https://github.com/PujaSridhar/PujaSridhar.github.io.git", "/kaggle/working/repo"], check=True)
    shutil.copytree("/kaggle/working/repo/cogsworth/train", WORK, ignore=shutil.ignore_patterns("node_modules"))
    REPO = Path("/kaggle/working/repo")
os.environ["REPO_SRC"] = str(REPO or "")
for f in ["train.jsonl", "val.jsonl", "eval-prompts.jsonl"]:
    print(f, sum(1 for _ in open(WORK / "data" / f)), "rows")

In [ ]:
%%bash
# Packages. torch comes with Kaggle's image (CUDA build), so it is not reinstalled.
# pip may print dependency warnings about gradio; they don't affect this notebook.
# Kaggle's preinstalled diffusers needs a newer huggingface_hub than transformers 4.x allows, and optimum's
# ONNX exporter imports it if present, so it is removed (nothing here uses it).
set -euo pipefail
pip uninstall -y -q diffusers || true
pip install -q -r /kaggle/working/cogsworth-train/python/requirements.txt
nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Fine-tune (LoRA, fp16 on the T4)

Only the assistant replies are trained on; the system prompt, retrieved context and question are masked.
Watch `eval_loss` after each epoch: it should drop and then flatten. The best epoch is kept.

In [ ]:
%%bash
# Each shell cell stops the run on the first error (set -e), so a failed step can't be skipped silently.
set -euo pipefail
cd /kaggle/working/cogsworth-train/python
python train_lora.py --base "$BASE" --data ../data --out /kaggle/working/out --epochs "$EPOCHS" \
  --precision fp16 --batch 1 --grad-accum 16

## 2. Answer the eval questions

Writes answers from the trained model and from the untuned base model, so you can see what training
changed. Score them on your laptop with `node cogsworth/train/score.js` (see the README).

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/cogsworth-train/python
python generate.py --base "$BASE" --adapter /kaggle/working/out/adapter --out /kaggle/working/out/trained-answers.jsonl --label "trained model"
python generate.py --base "$BASE" --out /kaggle/working/out/base-answers.jsonl --label "base model"

In [ ]:
# A quick look at a few answers
import json
rows = [json.loads(l) for l in open("/kaggle/working/out/trained-answers.jsonl")]
base = {(r["suite"], r["index"]): r["answer"] for r in map(json.loads, open("/kaggle/working/out/base-answers.jsonl"))}
for r in rows[:12]:
    print("Q:", r["q"])
    print("  trained:", r["answer"])
    print("  base:   ", base.get((r["suite"], r["index"]), "")[:300])
    print()

## 3. Export for the browser

Merges the adapter into the base model, exports ONNX with optimum, and quantizes to q8 and q4.
The folder `cogsworth-web` loads directly with transformers.js.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/cogsworth-train/python
python export_onnx.py --base "$BASE" --adapter /kaggle/working/out/adapter --out /kaggle/working/out/cogsworth-web --formats "$EXPORT_FORMATS"

In [ ]:
%%bash
# Package the results. The merged fp32 model and checkpoints are left out (large, and rebuildable from the adapter).
set -euo pipefail
cd /kaggle/working/out
rm -rf checkpoints merged onnx-fp32
zip -qr /kaggle/working/cogsworth-out.zip adapter cogsworth-web trained-answers.jsonl base-answers.jsonl training-summary.json
ls -lh /kaggle/working/cogsworth-out.zip
echo "Download cogsworth-out.zip from the Output tab, then follow 'After training' in cogsworth/train/README.md"

## 4. Score it

Runs `score.js` here: the engine, the untuned base model, the trained model, and the exported browser model
(q8, with and without the grounding guard) on all 142 eval questions. The table prints below, and the
side-by-side `report.md` is saved next to the zip. Needs the repo itself as the input (a GitHub-repo dataset),
or no input at all so the notebook clones it.

In [ ]:
%%bash
set -euo pipefail
if [ -z "$REPO_SRC" ]; then echo "The input has only the kit, not the whole repo, so scoring is skipped. Score on your laptop (README)."; exit 0; fi
NODE=v22.12.0
if ! command -v node >/dev/null || [ "$(node -p 'process.versions.node.split(".")[0]')" -lt 22 ]; then
  curl -fsSL https://nodejs.org/dist/$NODE/node-$NODE-linux-x64.tar.xz | tar -xJ -C /tmp
  export PATH=/tmp/node-$NODE-linux-x64/bin:$PATH
fi
node --version
S=/kaggle/working/score
rm -rf $S && mkdir -p $S
cp "$REPO_SRC/package.json" "$REPO_SRC/portfolio-data.js" $S/
cp -r "$REPO_SRC/cogsworth" $S/ && rm -rf $S/cogsworth/node_modules $S/cogsworth/train/python/out
npm --prefix $S/cogsworth install --no-audit --no-fund --loglevel=error
cd $S
node cogsworth/train/score.js --base /kaggle/working/out/base-answers.jsonl --trained /kaggle/working/out/trained-answers.jsonl \
  --onnx /kaggle/working/out/cogsworth-web --dtype q8
cp cogsworth/train/eval/out/report.md /kaggle/working/report.md